# V029 synthetic preflight
Explicit CPU/CUDA checks only; definitions read no assignment data. Test native MAE selection and exact checkpoint recovery; preserve frozen base notebooks.

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,copy,time,gc
import numpy as np
import torch

def run_laplace_preflight_v029(project_root,output_directory,device='cpu',deep_spec=None):
    P=Path(project_root).resolve();O=Path(output_directory);O.mkdir(parents=True,exist_ok=False)
    checks={}
    def check(name,value):
        if not bool(value):raise AssertionError(name)
        checks[name]=True
    def code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
    def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
    def equal(a,b):
        if torch.is_tensor(a):return torch.is_tensor(b) and torch.equal(a,b)
        if isinstance(a,np.ndarray):return isinstance(b,np.ndarray) and np.array_equal(a,b)
        if isinstance(a,dict):return isinstance(b,dict) and a.keys()==b.keys() and all(equal(a[k],b[k]) for k in a)
        if isinstance(a,(list,tuple)):return type(a)==type(b) and len(a)==len(b) and all(equal(x,y) for x,y in zip(a,b))
        return a==b
    ns={'__name__':'ventilator_v029_preflight'};sources={}
    for name in ['feature_components_v005.ipynb','sequence_components_v005.ipynb','training_components_v008.ipynb','laplace_components_v029.ipynb']:
        text=code(P/'research'/name);sources[name]=hashlib.sha256(text.encode()).hexdigest();exec(compile(text,name,'exec'),ns)
    selected=ns['_sequence_device'](device);ns['seed_sequence'](20261102)
    check('actual_requested_device',selected.type==device)
    p=torch.tensor([[.2,-.4,.9,.1]],device=selected,requires_grad=True)
    target=torch.tensor([[.1,.2,-9.,-.4]],device=selected)
    mask=torch.tensor([[True,True,False,True]],device=selected)
    b=torch.ones_like(p,requires_grad=True)
    objective,nll,adjacent=ns['laplace_curve_loss_v029'](p,target,b,mask)
    original=ns['curve_l1_loss_v008'](p,target,mask)[0]
    check('frozen_unit_scale_matches_original_curve_loss',torch.equal(objective,original))
    g=torch.autograd.grad(objective,(p,b),retain_graph=True);g0=torch.autograd.grad(original,p,retain_graph=True)[0]
    check('frozen_unit_scale_matches_original_mean_gradient',torch.equal(g[0],g0))
    check('expired_mean_output_has_zero_objective_gradient',bool((g[0][~mask]==0).all()))
    check('expired_scale_output_has_zero_objective_gradient',bool((g[1][~mask]==0).all()))
    expected_b=torch.where(mask,(1-(p-target).abs())/mask.sum(),torch.zeros_like(p))
    check('analytic_scale_gradient_correct',torch.allclose(g[1],expected_b,rtol=1e-6,atol=1e-7))
    changed=target.masked_fill(~mask,10000.)
    check('expired_pressure_does_not_change_likelihood_or_curve',torch.equal(ns['laplace_curve_loss_v029'](p,changed,b,mask)[0],objective))
    check('only_adjacent_inspiratory_pairs_penalized',torch.allclose(adjacent,p.new_tensor(.7)))
    isolated=torch.tensor([[True,False,True,False]],device=selected)
    check('empty_adjacent_pair_term_finite_zero',float(ns['laplace_curve_loss_v029'](p,target,b,isolated)[2])==0.)
    varying=torch.tensor([[.5,2.,3.,.7]],device=selected,requires_grad=True)
    manual=((p-target).abs()/varying+varying.log())[mask].mean()+.1*((p-target)[:,1:]-(p-target)[:,:-1]).abs()[(mask[:,1:]&mask[:,:-1])].mean()
    check('varying_positive_scale_likelihood_exact',torch.equal(ns['laplace_curve_loss_v029'](p,target,varying,mask)[0],manual))
    try:ns['laplace_curve_loss_v029'](p,target,-b,mask)
    except ValueError:checks['nonpositive_scale_rejected']=True
    else:raise AssertionError('Nonpositive scale accepted')
    rng=np.random.default_rng(20262901);n_features=47;steps=np.arange(80,dtype=np.float32)[None,:]
    raw=rng.normal(0.,.2,(8,80,n_features)).astype(np.float32)
    raw[:,:,0]=np.array([5,20,50,5,20,50,5,20],dtype=np.float32)[:,None]
    raw[:,:,1]=np.array([10,20,50,20,50,10,50,10],dtype=np.float32)[:,None]
    raw[:,:,2]=np.broadcast_to(steps*.025,(8,80));raw[:,:,3]=8.+2*np.sin(steps/7.)+.3*np.arange(8)[:,None]
    out=np.broadcast_to(steps>=31,(8,80)).astype(np.int8).copy();raw[:,:,4]=out
    y=(3.+.13*raw[:,:,3]+.07*steps+.1*np.arange(8)[:,None]+.03*rng.normal(size=(8,80))).astype(np.float32)
    names=['R','C','time_step','u_in','u_out']+['toy_'+str(i) for i in range(5,n_features)]
    spec=dict(family='lstm',hidden=8,layers=2,dropout=.15,bidirectional=True,residual=True,widths=[8,6],output_concat=True,head_width=8)
    cfg=dict(lr=.0015,batch_size=2,max_epochs=3,patience=0,min_delta=0.,scheduler='replay',frozen_lr_schedule=[.0015,.001,.0005],prediction_batch_size=2,optimizer='Adam',weight_decay=0.,loss='l1')
    prep=ns['fit_preprocessor'](raw[:6],y[:6],out[:6],spec,names)
    transformed=ns['transform_inputs'](raw,prep);tensor=torch.from_numpy(transformed[:2]).to(selected)
    ns['seed_sequence'](20261102);plain=ns['V5PressureRNN'](spec,n_features).to(selected);rng_after_plain=torch.get_rng_state().clone()
    cuda_after_plain=torch.cuda.get_rng_state_all() if selected.type=='cuda' else None
    ns['seed_sequence'](20261102);model=ns['v029_laplace_factory'](spec,n_features,prep).to(selected)
    check('scale_head_initialization_consumes_no_CPU_RNG',torch.equal(torch.get_rng_state(),rng_after_plain))
    if selected.type=='cuda':check('scale_head_initialization_consumes_no_CUDA_RNG',equal(torch.cuda.get_rng_state_all(),cuda_after_plain))
    check('all_original_mean_parameters_exact_at_initialization',all(torch.equal(v,model.state_dict()[k]) for k,v in plain.state_dict().items()))
    plain.eval();model.eval()
    with torch.no_grad():a=plain(tensor);m=model(tensor);scale=model.v029_scale
    check('initial_mean_forward_exact_control',torch.equal(a,m))
    check('initial_scaled_positive_scale_is_unit',torch.allclose(scale,torch.ones_like(scale),rtol=0,atol=2e-7))
    plain.train();model.train();ns['seed_sequence'](20261103);a=plain(tensor);next_rng=torch.get_rng_state().clone();next_cuda=torch.cuda.get_rng_state_all() if selected.type=='cuda' else None
    ns['seed_sequence'](20261103);m=model(tensor)
    check('training_dropout_mean_forward_exact_control',torch.equal(a,m))
    check('training_dropout_CPU_RNG_stream_exact_control',torch.equal(next_rng,torch.get_rng_state()))
    if selected.type=='cuda':check('training_dropout_CUDA_RNG_stream_exact_control',equal(next_cuda,torch.cuda.get_rng_state_all()))
    check('input_only_scale_shape_positive',model.v029_scale.shape==m.shape and bool((model.v029_scale>0).all()))
    with torch.no_grad():
        model.head[-1].weight.zero_();model.head[-1].bias.fill_(-10.);negative=model(tensor)
    check('mean_path_allows_negative_pressure_without_clipping',bool((negative<0).all()))
    del model,plain;gc.collect()
    if selected.type=='cuda':torch.cuda.empty_cache()
    common=dict(model_spec=spec,train_cfg=cfg,seed=20261102,feature_names=names,train_diagnostic_breaths=4,
        device=device,model_factory=ns['v029_laplace_factory'],loss_kind='laplace_curve',arm='v029_synthetic',source_fingerprints=sources)
    class Interrupted(RuntimeError):pass
    def interrupt(record):
        if record['epoch']==1:raise Interrupted('Saved epoch-boundary synthetic interruption')
    summaries={}
    for mode in ['validate','fixed']:
        full=O/(mode+'-full.pt');resume_path=O/(mode+'-resume.pt')
        def run(path,resume=False,on_epoch=None,**overrides):
            kw=dict(common,checkpoint_path=path,resume=resume,on_epoch=on_epoch);kw.update(overrides)
            if mode=='validate':return ns['fit_validate_v029'](raw[:6],y[:6],out[:6],raw[6:],y[6:],out[6:],**kw)
            kw.pop('train_diagnostic_breaths');return ns['fit_fixed_v029'](raw[:6],y[:6],out[:6],lr_schedule=cfg['frozen_lr_schedule'],**kw)
        full_report=run(full)
        try:run(resume_path,on_epoch=interrupt)
        except Interrupted:checks[mode+'_interrupt_after_authoritative_checkpoint']=True
        else:raise AssertionError('No synthetic interruption')
        resumed_report=run(resume_path,resume=True)
        aa=torch.load(ns['_last_path'](full),map_location='cpu',weights_only=False);bb=torch.load(ns['_last_path'](resume_path),map_location='cpu',weights_only=False)
        for field in ['state_dict','optimizer_state','rng_state','scheduler_state','preprocess','fingerprint','model_spec','config','epoch','optimizer_steps']:
            check(mode+'_exact_resume_'+field,equal(aa[field],bb[field]))
        fields=['lr_used','training_online_inspiratory_mae','training_online_objective_scaled','training_online_laplace_NLL_scaled','training_online_increment_mae','training_online_scale_mean_native']
        if mode=='validate':fields+=['validation_inspiratory_mae','best_val_mae','best_epoch']
        check(mode+'_exact_epoch_metrics',all(all(equal(a[f],b[f]) for f in fields) for a,b in zip(full_report['epochs'],resumed_report['epochs'])))
        options=dict(device=device,batch_size=2,source_fingerprints=sources,expected_arm=common['arm'],expected_loss_kind='laplace_curve')
        mu,scale=ns['predict_checkpoint_laplace_v029'](full,raw[6:],return_scale=True,**options)
        rm,rb=ns['predict_checkpoint_laplace_v029'](resume_path,raw[6:],return_scale=True,**options)
        check(mode+'_native_mean_and_scale_exact_resume',np.array_equal(mu,rm) and np.array_equal(scale,rb))
        check(mode+'_native_prediction_finite_positive_scale',mu.shape==scale.shape==(2,80) and np.isfinite(mu).all() and np.isfinite(scale).all() and (scale>0).all())
        check(mode+'_truthful_primary_objective_metadata',full_report['loss_kind']=='laplace_curve' and full_report['actual_training_objective']=='laplace_curve' and aa['v029']['objective']['loss_kind']=='laplace_curve' and aa['actual_training_objective']=='laplace_curve')
        if mode=='validate':
            native=ns['inspiratory_mae'](y[6:],mu,out[6:])
            check('native_MAE_best_checkpoint_selection',full_report['best_epoch']==int(np.argmin([r['validation_inspiratory_mae'] for r in full_report['epochs']]))+1 and abs(native-full_report['validation_mae'])<1e-15)
        for label,override in [('arm',dict(arm='v029_changed')),('loss',dict(loss_kind='curve_l1')),('source',dict(source_fingerprints=dict(sources,changed_source='0'*64)))]:
            try:run(resume_path,resume=True,**override)
            except ValueError:checks[mode+'_reject_changed_'+label]=True
            else:raise AssertionError('Changed identity accepted: '+label)
        before=copy.deepcopy(ns['V029_OBJECTIVE']);ns['V029_OBJECTIVE']['scale_floor_target_scaled']=.002
        try:
            try:run(resume_path,resume=True)
            except ValueError:checks[mode+'_reject_changed_objective_configuration']=True
            else:raise AssertionError('Changed objective accepted')
        finally:ns['V029_OBJECTIVE'].clear();ns['V029_OBJECTIVE'].update(before)
        summaries[mode]=dict(executed_epochs=full_report['executed_epochs'],checkpoint_sha256=sha(full),last_sha256=sha(ns['_last_path'](full)))
        del aa,bb;gc.collect()
    workflow=ns['v029_workflow_namespace'](P);policy_path=P/'research/pruning_policy_v025.json';policy_sha=sha(policy_path)
    job=dict(id='v029_queue_synthetic_representative',variant='laplace_curve',fit_args=(raw[:6],y[:6],out[:6],raw[6:],y[6:],out[6:],spec,cfg,20261102,O/'queue-model.pt'),fit_kwargs=dict(feature_names=names,device=device,model_factory=ns['v029_laplace_factory'],loss_kind='laplace_curve',arm='v029_queue_synthetic',source_fingerprints=sources,train_diagnostic_breaths=4))
    queued=workflow['run_training_queue_v025']([job],project_root=P,policy_path=policy_path,expected_policy_sha256=policy_sha,state_path=O/'queue-state.json')
    check('V025_queue_executes_full_unfamiliar_objective',queued[0]['status']=='completed' and queued[0]['report']['executed_epochs']==3)
    check('V025_identity_truthful_objective',queued[0]['identity']['cohort']['objective']==ns['V029_OBJECTIVE'])
    decisions=[json.loads(l) for l in (O/'queue-model.pruning.jsonl').read_text().splitlines()]
    check('unfamiliar_objective_not_aggressively_pruned',len(decisions)==3 and all(v['action']=='continue' for v in decisions))
    pred=workflow['predict_completed_v025'](queued[0],raw[6:],device=device,batch_size=2)
    check('private_workflow_completed_prediction_truthful_loader',pred.shape==(2,80) and np.isfinite(pred).all())
    check('original_V008_trainer_binding_unchanged',ns['fit_validate_v008'] is not ns['fit_validate_v029'])
    profiles=[]
    if deep_spec is not None:
        ns['seed_sequence'](20261102);plain=ns['V5PressureRNN'](deep_spec,n_features).to(selected)
        ns['seed_sequence'](20261102);deep=ns['v029_laplace_factory'](deep_spec,n_features,prep).to(selected)
        plain.eval();deep.eval()
        with torch.no_grad():bp=plain(tensor);dp=deep(tensor)
        check('full_deep_backbone_initial_mean_exact',torch.equal(bp,dp) and dp.shape==(2,80))
        check('full_deep_scale_finite_positive',bool(torch.isfinite(deep.v029_scale).all()) and bool((deep.v029_scale>0).all()))
        del plain,deep;gc.collect()
        timing_target=ns['_scaled_targets'](raw[:1],y[:1],prep)
        check('full_model_timing_target_helper_exercised_on_CPU_and_CUDA',timing_target.shape==(1,80) and timing_target.dtype==np.float32 and np.isfinite(timing_target).all())
        if selected.type=='cuda':
            torch.cuda.empty_cache();batch=256
            tx=torch.from_numpy(np.repeat(transformed[:1],batch,axis=0)).to(selected);ty=torch.from_numpy(np.repeat(timing_target,batch,axis=0)).to(selected);tm=torch.from_numpy(np.repeat(out[:1]==0,batch,axis=0)).to(selected)
            for name,factory in [('plain',None),('laplace',ns['v029_laplace_factory'])]:
                ns['seed_sequence'](20261102);m=ns['V5PressureRNN'](deep_spec,n_features) if factory is None else factory(deep_spec,n_features,prep);m=m.to(selected);opt=torch.optim.Adam(m.parameters(),lr=.0015)
                times=[]
                for repeat in range(4):
                    torch.cuda.synchronize();started=time.perf_counter();opt.zero_grad(set_to_none=True);mean=m(tx)
                    loss=ns['curve_l1_loss_v008'](mean,ty,tm)[0] if name=='plain' else ns['laplace_curve_loss_v029'](mean,ty,m.v029_scale,tm)[0]
                    loss.backward();nn=ns['nn'];nn.utils.clip_grad_norm_(m.parameters(),1.);opt.step();torch.cuda.synchronize()
                    if repeat>0:times.append(time.perf_counter()-started)
                profiles.append(dict(name=name,batch_size=batch,median_step_seconds=float(np.median(times)),peak_allocated_bytes=int(torch.cuda.max_memory_allocated())))
                del m,opt,mean,loss;gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
            profiles[1]['step_time_ratio_to_baseline']=profiles[1]['median_step_seconds']/profiles[0]['median_step_seconds']
            check('full_model_CUDA_efficiency_bounded',profiles[1]['step_time_ratio_to_baseline']<1.7 and profiles[1]['peak_allocated_bytes']<10*1024**3)
    result=dict(status='passed',created_utc=datetime.now(timezone.utc).isoformat(),checks=checks,check_count=len(checks),source_code_hashes=sources,
        actual_cuda=selected.type=='cuda',exact_resume=True,native_inference=True,raw_approved_csv_reads=0,approved_data_fits=0,
        synthetic_only=True,objective=copy.deepcopy(ns['V029_OBJECTIVE']),summaries=summaries,efficiency_profiles=profiles,
        runtime=dict(torch=str(torch.__version__),cuda=torch.version.cuda,numpy=np.__version__,device=device))
    (O/'proof.json').write_text(json.dumps(result,indent=2)+'\n');print(json.dumps(result,indent=2));return result


{
  "status": "passed",
  "created_utc": "2026-11-08T05:47:45.691726+00:00",
  "checks": {
    "actual_requested_device": true,
    "frozen_unit_scale_matches_original_curve_loss": true,
    "frozen_unit_scale_matches_original_mean_gradient": true,
    "expired_mean_output_has_zero_objective_gradient": true,
    "expired_scale_output_has_zero_objective_gradient": true,
    "analytic_scale_gradient_correct": true,
    "expired_pressure_does_not_change_likelihood_or_curve": true,
    "only_adjacent_inspiratory_pairs_penalized": true,
    "empty_adjacent_pair_term_finite_zero": true,
    "varying_positive_scale_likelihood_exact": true,
    "nonpositive_scale_rejected": true,
    "scale_head_initialization_consumes_no_CPU_RNG": true,
    "all_original_mean_parameters_exact_at_initialization": true,
    "initial_mean_forward_exact_control": true,
    "initial_scaled_positive_scale_is_unit": true,
    "training_dropout_mean_forward_exact_control": true,
    "training_dropout_CPU_RNG_stream_e